[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/verification/schafer-turek-2d1.ipynb)

# **Schäfer-Turek 2D-1 benchmark**
**Johan Hoffman**

# **Abstract**

The GLS stabilized Navier-Stokes solver is applied to the Schäfer-Turek 2D-1 benchmark (steady Re=20 flow past a cylinder in a channel, Schäfer & Turek 1996). Two of three refinement levels are fully computed (level 3, resolution=128, is stopped by the 1-hour wall-time limit). A steady-state stopping criterion (`‖u1−u0‖/dt/‖u1‖ < 1e−6`) terminates time integration at t ≈ 19.5 and t ≈ 20.3 for levels 1 and 2 respectively. Drag coefficient C_D converges from 9.6% to 2.6% error and pressure drop Δp from 3.9% to 2.0% error relative to the reference values. Lift coefficient C_L has large relative errors at both levels because the reference value (C_L = 0.0106) is nearly zero due to the near-symmetric geometry.

# **About the code**

In [ ]:
# Copyright (C) 2020-2026 Johan Hoffman (jhoffman@kth.se)
# SPDX-License-Identifier: MIT
#
# Verification notebook: Schäfer-Turek 2D-1 steady benchmark.
# This template is maintained by Johan Hoffman.
# Please report problems to jhoffman@kth.se

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import numpy as np
import time
import ufl
import basix.ufl as bufl
from mpi4py import MPI
from matplotlib import pyplot as plt

import dolfinx
from dolfinx import fem
from dolfinx.fem import functionspace, Function, form, assemble_scalar, Constant
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc

In [ ]:
# --- canonical: gmsh_rect_minus_circles v5 ---
import math
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


_ALPHA = 0.66
# mshr numSamples=100 → N = floor(sqrt(100.5)) = 10 interior sample rows
_N_SAMPLE = int(math.floor(math.sqrt(100.0 + 0.5)))  # 10


def _mshr_segments(r, L, H, resolution):
    """Exact mshr auto-segment count for Circle(segments=0) in 2D.

    Source path:
      MeshGenerator.cpp: segment_granularity = 2*R_enc/resolution
      CSGGeometry.cpp:   estimate_bounding_sphere samples _N_SAMPLE×_N_SAMPLE
                         interior points; extreme corners dominate, giving
                         R_enc = ((N-1)/N) * sqrt(L²+H²) / 2
      CSGCGALDomain2D.cpp: num_segs = max(5, round(2π*r / segment_granularity))

    Verified predictions (no Steiner points since edge_len < cs_mshr):
      4×2 domain, r=0.2, resolution=32 → 10
      4×4 domain, r=0.2, resolution=32 →  8
    """
    R_enc = ((_N_SAMPLE - 1) / _N_SAMPLE) * math.sqrt(L ** 2 + H ** 2) / 2.0
    cs = 2.0 * R_enc / resolution
    return max(5, round(2.0 * math.pi * r / cs))


def gmsh_rect_minus_circles(L, H, circles, resolution, segments=32):
    """Rectangle [0,L]×[0,H] minus regular-polygon holes.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii.
    resolution : int
        Mesh density parameter.  Interior mesh size:
            lc = _ALPHA * sqrt(L² + H²) / resolution
    segments : int or "mshr"
        Number of polygon sides used to approximate each circle hole.
        Default 32.  Pass ``"mshr"`` to use the legacy mshr auto-segment
        rule (max(5, round(2π·r/cs)), cs derived from bounding-sphere
        estimate), which gives 10 sides for 4×2 domains and 8 for 4×4.

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Notes
    -----
    Vertex placement: phi_i = 2π·i/n, i = 0 … n-1 (i=0 at angle 0, CCW).

    Mesh sizing: one Distance+Threshold gmsh field per polygon transitions
    from ``edge_len = 2r·sin(π/n)`` right at the polygon boundary (prevents
    Steiner-point insertion on polygon edges) to ``lc`` at distance ``r/2``.
    No MeshSizeMax override; background field is the sole size control.
    """
    lc = _ALPHA * math.sqrt(L ** 2 + H ** 2) / resolution
    eps = 1e-6 * max(L, H)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)

    hole_tags = []
    segs_per_circle = []
    for cx, cy, r in circles:
        if segments == "mshr":
            n = _mshr_segments(r, L, H, resolution)
        elif isinstance(segments, int):
            n = segments
        else:
            raise ValueError(f"segments must be an int or 'mshr', got {segments!r}")
        segs_per_circle.append(n)
        pts = [
            gmsh.model.occ.addPoint(
                cx + r * math.cos(2 * math.pi * i / n),
                cy + r * math.sin(2 * math.pi * i / n),
                0,
            )
            for i in range(n)
        ]
        lines = [
            gmsh.model.occ.addLine(pts[i], pts[(i + 1) % n])
            for i in range(n)
        ]
        cl = gmsh.model.occ.addCurveLoop(lines)
        surf = gmsh.model.occ.addPlaneSurface([cl])
        hole_tags.append((2, surf))

    if hole_tags:
        gmsh.model.occ.cut([(2, 1)], hole_tags)
    gmsh.model.occ.synchronize()

    # Distance+Threshold field per polygon hole
    bg_fields = []
    for i, (cx, cy, r) in enumerate(circles):
        n = segs_per_circle[i]
        edge_len = 2 * r * math.sin(math.pi / n)

        # Identify polygon curves (not on the rectangle boundary)
        c_curves = []
        for dim, tag in gmsh.model.getEntities(1):
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if xmin <= eps or xmax >= L - eps or ymin <= eps or ymax >= H - eps:
                continue
            bounds = gmsh.model.getParametrizationBounds(dim, tag)
            mid = float(np.array(bounds).mean())
            x, y, _ = gmsh.model.getValue(dim, tag, [mid])
            if abs(math.hypot(x - cx, y - cy) - r) < 0.15 * r:
                c_curves.append(tag)

        if not c_curves:
            continue

        d_id = 10 + 2 * i
        t_id = 11 + 2 * i
        gmsh.model.mesh.field.add("Distance", d_id)
        gmsh.model.mesh.field.setNumbers(d_id, "CurvesList", c_curves)
        gmsh.model.mesh.field.add("Threshold", t_id)
        gmsh.model.mesh.field.setNumber(t_id, "InField", d_id)
        # SizeMin = edge_len at polygon: prevents subdivision of polygon edges.
        # SizeMax = lc at distance r/2: interior mesh size.
        gmsh.model.mesh.field.setNumber(t_id, "SizeMin", edge_len)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMax", lc)
        gmsh.model.mesh.field.setNumber(t_id, "DistMin", 0.0)
        gmsh.model.mesh.field.setNumber(t_id, "DistMax", r / 2.0)
        bg_fields.append(t_id)

    if bg_fields:
        if len(bg_fields) == 1:
            gmsh.model.mesh.field.setAsBackgroundMesh(bg_fields[0])
        else:
            min_id = 10 + 2 * len(circles)
            gmsh.model.mesh.field.add("Min", min_id)
            gmsh.model.mesh.field.setNumbers(min_id, "FieldsList", bg_fields)
            gmsh.model.mesh.field.setAsBackgroundMesh(min_id)

    # Background field is the sole size control
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm", 5)  # Delaunay

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")
    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

# **Introduction**

The Schäfer-Turek benchmark describes steady laminar flow past a circular cylinder in a channel. The domain is [0, 2.2] × [0, 0.41] with a cylinder of radius 0.05 centred at (0.2, 0.2). A parabolic inflow profile is prescribed at the left boundary. Reference quantities (Schäfer & Turek 1996): drag coefficient C_D = 5.57953523384, lift coefficient C_L = 0.010618948146, pressure drop Δp = 0.11752016697. Three mesh refinement levels (resolution=32,64,128 with segments=32,64,128) are used to assess convergence.

# **Method**

**Parameters**

In [ ]:
# Problem geometry
L, H = 2.2, 0.41
cx, cy, r = 0.2, 0.2, 0.05

# Physical parameters
nu = 1e-3       # kinematic viscosity
U_m = 0.3       # max inflow velocity; mean U = 2/3·U_m

# Simulation time and convergence
T = 30.0        # max integration time (Re=20, convective scale ~11s)
plot_freq = 5
tol_ss = 1e-6   # steady-state criterion: ||u1-u0||/dt/||u1|| < tol_ss

# Reference values (Schäfer & Turek 1996, 2D-1 steady benchmark)
C_D_ref = 5.57953523384
C_L_ref = 0.010618948146
dp_ref  = 0.11752016697

# Refinement levels: coarse, medium, fine
levels = [
    {"resolution": 32,  "segments": 32},
    {"resolution": 64,  "segments": 64},
    {"resolution": 128, "segments": 128},
]

**Convergence study**

In [ ]:
from dolfinx.geometry import (
    bb_tree as _bb_tree,
    compute_collisions_points as _ccp,
    compute_colliding_cells as _ccc,
)

U_mean    = (2.0 / 3.0) * U_m
D_cyl     = 2.0 * r
force_ref = -2.0 / (U_mean**2 * D_cyl)   # C = force_ref * Force_form (rho=1)

results = []

for lvl_idx, lvl in enumerate(levels):
    if lvl_idx > 0 and T < 1.0:   # fast mode: coarsest level only
        results.append(None)
        continue

    resolution = lvl["resolution"]
    segs       = lvl["segments"]
    print(f"\n--- Level {lvl_idx + 1}: resolution={resolution}, segments={segs} ---",
          flush=True)

    # Mesh
    msh, _ = gmsh_rect_minus_circles(L, H, [(cx, cy, r)], resolution, segments=segs)
    facet_tags = tag_boundaries(msh, L, H)

    tdim    = msh.topology.dim
    fdim    = tdim - 1
    n_cells = msh.topology.index_map(tdim).size_local
    h_arr   = msh.h(tdim, np.arange(n_cells, dtype=np.int32))
    hmin    = float(h_arr.min())
    print(f"  cells={n_cells}, hmin={hmin:.6f}", flush=True)

    plot_mesh(msh, title=f"Level {lvl_idx + 1}: res={resolution}, segs={segs}, cells={n_cells}")

    # Spaces
    P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
    P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
    V   = functionspace(msh, P1v)
    Q   = functionspace(msh, P1s)
    V0, _ = V.sub(0).collapse()
    V1, _ = V.sub(1).collapse()

    inlet_facets  = facet_tags.find(1)
    outlet_facets = facet_tags.find(2)
    lower_facets  = facet_tags.find(3)
    upper_facets  = facet_tags.find(4)
    cyl_facets    = facet_tags.find(5)
    wall_facets   = np.concatenate([lower_facets, upper_facets])

    # Boundary conditions
    def _zero_bc(sub_idx, sub_V, facets):
        f = Function(sub_V); f.x.array[:] = 0.0
        dofs = fem.locate_dofs_topological((V.sub(sub_idx), sub_V), fdim, facets)
        return fem.dirichletbc(f, dofs, V.sub(sub_idx))

    u_in_func = Function(V)
    u_in_func.interpolate(lambda x: np.vstack([
        4.0 * U_m * x[1] * (H - x[1]) / H**2,
        np.zeros_like(x[1]),
    ]))
    bcu_in   = fem.dirichletbc(u_in_func,
                               fem.locate_dofs_topological(V, fdim, inlet_facets))
    bcu_wall = [_zero_bc(0, V0, wall_facets), _zero_bc(1, V1, wall_facets)]
    bcu_cyl  = [_zero_bc(0, V0, cyl_facets),  _zero_bc(1, V1, cyl_facets)]
    bcu = [bcu_in] + bcu_wall + bcu_cyl
    bcp = [fem.dirichletbc(Constant(msh, PETSc.ScalarType(0.0)),
                           fem.locate_dofs_topological(Q, fdim, outlet_facets), Q)]

    # Functions and time step
    u0 = Function(V, name="u0"); u1 = Function(V, name="u1")
    p1 = Function(Q, name="p1")
    dt_val = 0.5 * hmin
    dt_c   = Constant(msh, PETSc.ScalarType(dt_val))
    nu_c   = Constant(msh, PETSc.ScalarType(nu))

    # Variational forms — GLS stabilized NS (identical to course scheme)
    u = ufl.TrialFunction(V); v = ufl.TestFunction(V)
    p = ufl.TrialFunction(Q); q = ufl.TestFunction(Q)

    h_c   = ufl.CellDiameter(msh)
    u_mag = ufl.sqrt(ufl.dot(u1, u1))
    d1    = 1.0 / ufl.sqrt((1.0/dt_c)**2 + (u_mag/h_c)**2)
    d2    = h_c * u_mag
    um    = 0.5 * (u + u0)
    um1   = 0.5 * (u1 + u0)

    Fu = (ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1), v)*ufl.dx
        - p1*ufl.div(v)*ufl.dx
        + nu_c*ufl.inner(ufl.grad(um), ufl.grad(v))*ufl.dx
        + d1*ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1) + ufl.grad(p1),
                       ufl.dot(ufl.grad(v), um1))*ufl.dx
        + d2*ufl.div(um)*ufl.div(v)*ufl.dx)

    Fp = (d1*ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1) + ufl.grad(p),
                       ufl.grad(q))*ufl.dx
        + ufl.div(um1)*q*ufl.dx)

    a_u = form(ufl.lhs(Fu)); l_u = form(ufl.rhs(Fu))
    a_p = form(ufl.lhs(Fp)); l_p = form(ufl.rhs(Fp))

    # Force computation — Green's formula (volume form)
    def _make_psi(phi_x, phi_y):
        psi = Function(V); psi.x.array[:] = 0.0
        dx_dofs = fem.locate_dofs_topological((V.sub(0), V0), fdim, cyl_facets)
        dy_dofs = fem.locate_dofs_topological((V.sub(1), V1), fdim, cyl_facets)
        if len(dx_dofs): psi.x.array[dx_dofs[0]] = phi_x
        if len(dy_dofs): psi.x.array[dy_dofs[0]] = phi_y
        psi.x.scatter_forward()
        return psi

    def _make_force_form(psi_f):
        return form(
            ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1), psi_f)*ufl.dx
            - p1*ufl.div(psi_f)*ufl.dx
            + nu_c*ufl.inner(ufl.grad(um1), ufl.grad(psi_f))*ufl.dx
        )

    psi_drag = _make_psi(1.0, 0.0)
    psi_lift = _make_psi(0.0, 1.0)
    drag_form = _make_force_form(psi_drag)
    lift_form = _make_force_form(psi_lift)

    # Pressure at front/rear stagnation (polygon vertices for even segments)
    p_pts  = np.array([[cx - r, cy, 0.0],
                       [cx + r, cy, 0.0]])
    p_tree = _bb_tree(msh, msh.topology.dim)
    p_cand = _ccp(p_tree, p_pts)
    p_adj  = _ccc(msh, p_cand, p_pts)
    p_cidx = np.array(
        [p_adj.array[p_adj.offsets[i]] for i in range(len(p_pts))],
        dtype=np.int32,
    )

    # Linear solvers
    A_u = create_matrix(a_u); b_u = create_vector(V)
    A_p = create_matrix(a_p); b_p = create_vector(Q)

    def _ksp(A, pc_type, amg=False):
        ksp = PETSc.KSP().create(msh.comm)
        ksp.setOperators(A); ksp.setType("bcgs")
        pc = ksp.getPC(); pc.setType(pc_type)
        if amg: pc.setHYPREType("boomeramg")
        ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); ksp.setFromOptions()
        return ksp

    ksp_u = _ksp(A_u, "ilu")
    ksp_p = _ksp(A_p, "hypre", amg=True)

    set_bc(u0.x.petsc_vec, bcu); u0.x.scatter_forward()
    set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

    # Time stepping
    t_wall0 = time.time()
    t = dt_val; step = 0; plot_t = 0.0
    t_stop = None
    _ss_met = False

    while t < T + 1e-10:
        for _ in range(5):
            A_u.zeroEntries()
            assemble_matrix(A_u, a_u, bcs=bcu); A_u.assemble()
            with b_u.localForm() as loc: loc.set(0.0)
            assemble_vector(b_u, l_u)
            apply_lifting(b_u, [a_u], bcs=[bcu])
            b_u.ghostUpdate(addv=PETSc.InsertMode.ADD,
                            mode=PETSc.ScatterMode.REVERSE)
            set_bc(b_u, bcu)
            ksp_u.setOperators(A_u, A_u); ksp_u.solve(b_u, u1.x.petsc_vec)
            u1.x.scatter_forward()
            set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

            A_p.zeroEntries()
            assemble_matrix(A_p, a_p, bcs=bcp); A_p.assemble()
            with b_p.localForm() as loc: loc.set(0.0)
            assemble_vector(b_p, l_p)
            apply_lifting(b_p, [a_p], bcs=[bcp])
            b_p.ghostUpdate(addv=PETSc.InsertMode.ADD,
                            mode=PETSc.ScatterMode.REVERSE)
            set_bc(b_p, bcp)
            ksp_p.setOperators(A_p, A_p); ksp_p.solve(b_p, p1.x.petsc_vec)
            p1.x.scatter_forward()
            set_bc(p1.x.petsc_vec, bcp); p1.x.scatter_forward()

        if t > plot_t:
            print(f"  t = {t:.3f}  ss_res={ss_res if step > 0 else float('nan'):.2e}", flush=True)
            plot_scalar(p1, title=f"Pressure, t={t:.1f}")
            plot_vector(u1, title=f"Velocity, t={t:.1f}")
            plot_t += T / plot_freq

        # Steady-state criterion: ||u1-u0|| / (dt * ||u1||)
        diff_arr = u1.x.array - u0.x.array
        nd = float(msh.comm.allreduce(np.dot(diff_arr, diff_arr), op=MPI.SUM)) ** 0.5
        n1 = float(msh.comm.allreduce(np.dot(u1.x.array, u1.x.array), op=MPI.SUM)) ** 0.5
        ss_res = nd / (dt_val * max(n1, 1e-30))

        u0.x.array[:] = u1.x.array
        t += dt_val; step += 1

        if ss_res < tol_ss:
            t_stop = t - dt_val
            print(f"  SS criterion met: t={t_stop:.4f}, ss_res={ss_res:.2e}", flush=True)
            _ss_met = True
            break

        # ~1-hour wall-time projection (after 5-step warmup)
        if step >= 5:
            elapsed = time.time() - t_wall0
            n_total = T / dt_val
            projected = elapsed * n_total / step
            if projected > 3600.0:
                t_stop = t - dt_val
                print(f"  Stopping: projected {projected/60:.0f} min > 60 min", flush=True)
                break

    if t_stop is None:
        t_stop = t - dt_val  # reached T without SS convergence

    t_wall = time.time() - t_wall0
    print(f"  Done: {step} steps, t_stop={t_stop:.4f}, wall={t_wall:.1f}s", flush=True)

    # QoIs at final step
    F_drag = float(msh.comm.allreduce(assemble_scalar(drag_form), op=MPI.SUM))
    F_lift = float(msh.comm.allreduce(assemble_scalar(lift_form), op=MPI.SUM))
    C_D = force_ref * F_drag
    C_L = force_ref * F_lift

    p_vals = p1.eval(p_pts, p_cidx)
    dp = float(p_vals[0, 0] - p_vals[1, 0])

    results.append({
        "lvl": lvl_idx + 1, "res": resolution, "segs": segs, "cells": n_cells,
        "hmin": hmin, "dt": dt_val, "steps": step, "t_stop": t_stop,
        "C_D": C_D, "C_L": C_L, "dp": dp, "wall": t_wall,
    })
    print(f"  C_D={C_D:.8f}  C_L={C_L:.10f}  Δp={dp:.10f}", flush=True)

# **Results**

In [ ]:
print("Schäfer-Turek 2D-1 convergence table")
print(f"Reference: C_D={C_D_ref}  C_L={C_L_ref}  Δp={dp_ref}")
print()
hdr = (f"{'lvl':>3} {'res':>4} {'cells':>7} {'hmin':>8} {'dt':>8} {'steps':>6} {'t_stop':>7}  "
       f"{'C_D':>12} {'err%':>7}  "
       f"{'C_L':>14} {'err%':>7}  "
       f"{'Δp':>12} {'err%':>7}  "
       f"{'wall':>8}")
print(hdr)
print("-" * len(hdr))
for res in results:
    if res is None:
        continue
    e_D  = 100.0 * abs(res["C_D"] - C_D_ref) / abs(C_D_ref)
    e_L  = 100.0 * abs(res["C_L"] - C_L_ref) / abs(C_L_ref)
    e_dp = 100.0 * abs(res["dp"]  - dp_ref)   / abs(dp_ref)
    print(f"{res['lvl']:>3d} {res['res']:>4d} {res['cells']:>7d} {res['hmin']:>8.5f} {res['dt']:>8.5f} {res['steps']:>6d} {res['t_stop']:>7.3f}  "
          f"{res['C_D']:>12.6f} {e_D:>6.2f}%  "
          f"{res['C_L']:>14.8f} {e_L:>6.2f}%  "
          f"{res['dp']:>12.8f} {e_dp:>6.2f}%  "
          f"{res['wall']:>7.1f}s")

# **Discussion**

The drag coefficient and pressure drop converge toward the reference values of Schäfer & Turek (1996) as the mesh is refined from resolution=32 (1236 cells, C_D error 9.6%, Δp error 3.9%) to resolution=64 (4652 cells, C_D error 2.6%, Δp error 2.0%), consistent with first-order accuracy expected for P1 elements.

The lift coefficient C_L has large relative errors (3154% and 433% at levels 1–2) because the reference value C_L = 0.0106 is nearly zero: the cylinder is offset by only 0.005 from the channel centreline (0.2 vs 0.205), making C_L a near-cancellation quantity. The absolute discretization error on P1 meshes at these resolutions exceeds C_L itself; this is a known property of the coarse-mesh regime and does not indicate a formula error.

The steady-state criterion terminates integration at t ≈ 20, approximately 1.8 convective time units (L/U_mean ≈ 11 s) after the main transient has decayed. Level 3 (resolution=128, 18504 cells) is not computed: the wall-time projection after 5 steps indicates 116 minutes, which exceeds the 1-hour limit.